In [ ]:
!pip install -q roboflow tensorflow matplotlib seaborn scikit-learn

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=" ")

project = rf.workspace(
    "image-classification-y0lsy"
).project("cats--dogs-and-birds")

dataset = project.version(3).download("folder")

print("Dataset downloaded:", dataset.location)

In [ ]:
import os

DATASET_PATH = dataset.location

print(os.listdir(DATASET_PATH))

for folder in ["train", "valid", "test"]:
    path = os.path.join(DATASET_PATH, folder)

    if os.path.exists(path):
        print("\n", folder)

        for class_name in os.listdir(path):
            class_path = os.path.join(path, class_name)

            if os.path.isdir(class_path):
                print(
                    class_name,
                    len(os.listdir(class_path))
                )

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_PATH, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

valid_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_PATH, "valid"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_names=class_names,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_PATH, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_names=class_names,
    shuffle=False
)

print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    layers.Lambda(
        tf.keras.applications.mobilenet_v2.preprocess_input
    ),

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(
        len(class_names),
        activation="softmax"
    )
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=20
)

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy * 100, "%")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

y_true = np.concatenate([
    y.numpy() for x, y in test_ds
])

predictions = model.predict(test_ds)

y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [ ]:
# Training and validation accuracy
plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Training and validation loss
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=0.00001
    )
]

print("Callbacks created successfully!")

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Freeze earlier layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep Batch Normalization layers frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Compile with a smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Fine-tuning
history_fine = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    callbacks = callbacks
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy * 100, "%")

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import seaborn as sns

y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

cm = confusion_matrix(y_true, y_pred)

print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=range(len(class_names)),
    target_names=class_names,
    zero_division=0
))

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names,
    cmap="Blues"
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Animal Classification Confusion Matrix")
plt.show()

In [ ]:
print("TensorFlow:", tf.__version__)
print("Model loaded:", 'model' in globals())
print("Classes:", class_names if 'class_names' in globals() else "Not loaded")

In [ ]:
from google.colab import files
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Upload an image
uploaded = files.upload()

if uploaded:

    image_path = next(iter(uploaded.keys()))

    # Load image
    img = tf.keras.utils.load_img(
        image_path,
        target_size=(224, 224)
    )

    # Convert image to array
    img_array = tf.keras.utils.img_to_array(img)

    # Add batch dimension
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model.predict(
        img_array,
        verbose=0
    )[0]

    # Get predicted class
    predicted_index = np.argmax(predictions)
    predicted_class = class_names[predicted_index]

    confidence = predictions[predicted_index] * 100

    print("Predicted Animal:", predicted_class)
    print("Confidence:", round(float(confidence), 2), "%")

    # Display image
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"{predicted_class}: {confidence:.2f}%")
    plt.axis("off")
    plt.show()

    # Display all predictions
    print("\nAll Class Probabilities:")

    for name, score in zip(class_names, predictions):
        print(f"{name}: {score * 100:.2f}%")

In [ ]:
# Save the trained model

model.save("/content/animal_classifier.keras")

# Save class names
import json

with open("/content/class_names.json", "w") as f:
    json.dump(class_names, f)

print("Model saved successfully!")
print("Classes:", class_names)

In [ ]:
!pip install -q tensorflowjs

In [ ]:
model.save("/content/animal_classifier.h5")

print("Model saved successfully!")

In [ ]:
import os

print("KERAS MODEL:", os.path.exists("/content/animal_classifier.keras"))
print("CLASS NAMES:", os.path.exists("/content/class_names.json"))
print("H5 MODEL:", os.path.exists("/content/animal_classifier.h5"))

In [ ]:

import glob
import re

files = glob.glob(
    "/usr/local/lib/python*/dist-packages/tensorflowjs/**/*.py",
    recursive=True
)

replacements = {
    "object": "object",
    "bool": "bool",
    "str": "str",
    "int": "int",
    "float": "float",
    "complex": "complex"
}

for path in files:
    with open(path, "r") as f:
        content = f.read()

    for old, new in replacements.items():
        content = re.sub(
            rf"\bnp\.{old}\b",
            new,
            content
        )

    with open(path, "w") as f:
        f.write(content)

print("TensorFlow.js compatibility patch completed!")


In [ ]:
from google.colab import files

files.download("/content/animal_classifier.keras")
files.download("/content/class_names.json")

In [ ]:

import tensorflow as tf
import json
import os

# Import MobileNetV2 preprocessing function
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Register the custom preprocessing function
custom_objects = {
    "preprocess_input": preprocess_input
}

# Load trained model
model = tf.keras.models.load_model(
    "/content/animal_classifier.keras",
    custom_objects=custom_objects,
    compile=False
)

# Load animal class names
with open("/content/class_names.json", "r") as f:
    class_names = json.load(f)

print("MODEL LOADED SUCCESSFULLY!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Animal classes:", class_names)


In [ ]:
import os
import shutil
import json

folder = "/content/Animal_Classifier_Model"

os.makedirs(folder, exist_ok=True)

shutil.copy(
    "/content/animal_classifier.keras",
    folder + "/animal_classifier.keras"
)

shutil.copy(
    "/content/class_names.json",
    folder + "/class_names.json"
)

print("Final model folder created!")
print(os.listdir(folder))

In [ ]:
!cd /content && zip -r Animal_Classifier_Model.zip Animal_Classifier_Model

In [ ]:
from google.colab import files

files.download("/content/Animal_Classifier_Model.zip")